# GHCN-Daily SNWD global build notebook (1998–present)

This notebook builds a **global station-level snow-depth dataset** from **GHCN-Daily** for `SNWD` (snow depth), intended as a clean intermediate product for later comparison with GEOSldas output.

## Design choices

- Use NOAA/NCEI **GHCN-Daily `by_year` CSV files**, not per-station `.dly` files.
  - This is much better for a **global 1998–present** extraction because you can stream one year at a time.
- Filter to **`ELEMENT == "SNWD"`** during ingest.
- Keep the result in an **analysis-friendly partitioned Parquet dataset**:
  - `output/ghcn_snwd_parquet/year=1998/part-....parquet`
  - `output/ghcn_snwd_parquet/year=1999/part-....parquet`
  - ...
- Also write:
  - a compact **station metadata table**
  - a **station inventory / coverage summary**
  - a **manifest** with provenance and run settings

That leaves you with a large but accessible, columnar dataset that downstream routines can read efficiently by year, date range, or station subset.

## Notes

- `SNWD` in GHCN-Daily is **snow depth in mm**.
- The NOAA `by_year` files contain rows with fields:
  `ID, YYYYMMDD, ELEMENT, DATA_VALUE, MFLAG, QFLAG, SFLAG, OBS_TIME`.
- A blank `QFLAG` means the value did not fail NOAA's QA checks.
- `ghcnd-inventory.txt` lists the first and last years of **unflagged** data for each station-element pair.

## Recommended environment

This notebook uses:
- `pandas`
- `pyarrow`
- `requests`

Install if needed:
```bash
mamba install pandas pyarrow requests jupyterlab
```

In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import json
import io
import gzip
import shutil

import pandas as pd
import requests

## Configuration

Set the year range and output location here.

For a first shakedown test, set `START_YEAR = 2024` and `END_YEAR = 2025`.  
For the real run, use `1998` to the current year you want.

In [ ]:
# -----------------------------
# User configuration
# -----------------------------
START_YEAR = 1998
END_YEAR = 2026

BASE_URL = "https://www.ncei.noaa.gov/pub/data/ghcn/daily"
WORKDIR = Path("ghcn_snwd_build")
OUTDIR = WORKDIR / "output"
TMPDIR = WORKDIR / "tmp"
PARQUET_DIR = OUTDIR / "ghcn_snwd_parquet"

DROP_QFLAGGED = True          # recommended default
DROP_MISSING = True           # drop DATA_VALUE == -9999
OVERWRITE_EXISTING_YEAR = False

WORKDIR.mkdir(exist_ok=True)
OUTDIR.mkdir(exist_ok=True)
TMPDIR.mkdir(exist_ok=True)
PARQUET_DIR.mkdir(exist_ok=True)

print("WORKDIR:", WORKDIR.resolve())
print("PARQUET_DIR:", PARQUET_DIR.resolve())

## Download station metadata and inventory

These are small and worth keeping locally.

In [ ]:
def download_file(url: str, dest: Path, chunk_size: int = 1024 * 1024) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return dest

stations_txt = WORKDIR / "ghcnd-stations.txt"
inventory_txt = WORKDIR / "ghcnd-inventory.txt"
readme_txt = WORKDIR / "ghcnd-readme.txt"

download_file(f"{BASE_URL}/ghcnd-stations.txt", stations_txt)
download_file(f"{BASE_URL}/ghcnd-inventory.txt", inventory_txt)
download_file(f"{BASE_URL}/readme.txt", readme_txt)

print(stations_txt, stations_txt.stat().st_size)
print(inventory_txt, inventory_txt.stat().st_size)
print(readme_txt, readme_txt.stat().st_size)

In [ ]:
def read_ghcn_stations(stations_file: str | Path) -> pd.DataFrame:
    colspecs = [
        (0, 11),   # station_id
        (12, 20),  # lat
        (21, 30),  # lon
        (31, 37),  # elev
        (38, 40),  # state
        (41, 71),  # name
        (72, 75),  # gsn_flag
        (76, 79),  # hcn_crn_flag
        (80, 85),  # wmo_id
    ]
    names = [
        "station_id", "lat", "lon", "elev", "state", "name",
        "gsn_flag", "hcn_crn_flag", "wmo_id"
    ]

    df = pd.read_fwf(stations_file, colspecs=colspecs, names=names, dtype=str)
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df["elev"] = pd.to_numeric(df["elev"], errors="coerce")

    for c in ["station_id", "state", "name", "gsn_flag", "hcn_crn_flag", "wmo_id"]:
        df[c] = df[c].fillna("").str.strip()

    return df


def read_ghcn_inventory(inventory_file: str | Path) -> pd.DataFrame:
    colspecs = [(0, 11), (12, 20), (21, 30), (31, 35), (36, 40), (41, 45)]
    names = ["station_id", "lat", "lon", "element", "first_year", "last_year"]

    df = pd.read_fwf(
        inventory_file,
        colspecs=colspecs,
        names=names,
        dtype={
            "station_id": str,
            "lat": float,
            "lon": float,
            "element": str,
            "first_year": int,
            "last_year": int,
        },
    )
    return df

stations = read_ghcn_stations(stations_txt)
inventory = read_ghcn_inventory(inventory_txt)

stations.head(), inventory.head()

## Identify all stations with SNWD coverage overlapping the requested period

This does **not** yet mean each station has a dense record. It just means NOAA inventory says there is unflagged `SNWD` data overlapping the requested years.

In [ ]:
snwd_inventory = (
    inventory.query("element == 'SNWD'")
    .loc[lambda d: d["last_year"] >= START_YEAR]
    .loc[lambda d: d["first_year"] <= END_YEAR]
    .copy()
)

snwd_stations = (
    snwd_inventory
    .merge(
        stations[["station_id", "lat", "lon", "elev", "state", "name", "gsn_flag", "hcn_crn_flag", "wmo_id"]],
        on="station_id",
        how="left",
        suffixes=("_inv", "")
    )
)

print("Stations with SNWD inventory overlapping requested period:", len(snwd_stations))
snwd_stations.head()

## Save station-level metadata tables now

These are useful on their own and downstream routines can read them without touching the full observation archive.

In [ ]:
station_meta_file = OUTDIR / "ghcn_snwd_station_metadata.parquet"
station_inventory_file = OUTDIR / "ghcn_snwd_inventory.parquet"

snwd_stations.to_parquet(station_meta_file, index=False)
snwd_inventory.to_parquet(station_inventory_file, index=False)

print(station_meta_file)
print(station_inventory_file)

## Year-by-year ingest from NOAA `by_year` files

This is the key part.

For each year:
1. Download `YYYY.csv.gz`
2. Stream it in chunks
3. Keep only `ELEMENT == "SNWD"`
4. Optionally remove missing and QA-flagged values
5. Parse date
6. Write out to partitioned Parquet by `year`

### Why this route?
Because for **global 1998–present**, parsing all per-station `.dly` files is clumsy.  
The yearly CSVs are much easier to stream and parallelize later if needed.

In [ ]:
BY_YEAR_COLUMNS = [
    "station_id",
    "date_str",
    "element",
    "data_value",
    "mflag",
    "qflag",
    "sflag",
    "obs_time",
]

DTYPE_MAP = {
    "station_id": "string",
    "date_str": "string",
    "element": "string",
    "data_value": "Int32",
    "mflag": "string",
    "qflag": "string",
    "sflag": "string",
    "obs_time": "string",
}

def year_url(year: int) -> str:
    return f"{BASE_URL}/by_year/{year}.csv.gz"

def year_tmp_file(year: int) -> Path:
    return TMPDIR / f"{year}.csv.gz"

def year_partition_dir(year: int) -> Path:
    return PARQUET_DIR / f"year={year}"

def output_exists_for_year(year: int) -> bool:
    part_dir = year_partition_dir(year)
    return part_dir.exists() and any(part_dir.glob("*.parquet"))

In [ ]:
def download_year_file(year: int) -> Path:
    dest = year_tmp_file(year)
    if dest.exists():
        return dest
    return download_file(year_url(year), dest)

def normalize_flags(df: pd.DataFrame) -> pd.DataFrame:
    for c in ["mflag", "qflag", "sflag", "obs_time"]:
        df[c] = df[c].fillna("").astype("string").str.strip()
    return df

def process_one_year(
    year: int,
    chunk_rows: int = 2_000_000,
    verbose: bool = True,
) -> dict:
    if output_exists_for_year(year) and not OVERWRITE_EXISTING_YEAR:
        return {
            "year": year,
            "status": "skipped_existing",
            "rows_written": None,
            "stations_written": None,
            "files": len(list(year_partition_dir(year).glob("*.parquet")))
        }

    gz_file = download_year_file(year)
    part_dir = year_partition_dir(year)
    part_dir.mkdir(parents=True, exist_ok=True)

    # clear old parquet files if overwriting
    if OVERWRITE_EXISTING_YEAR:
        for p in part_dir.glob("*.parquet"):
            p.unlink()

    total_rows_written = 0
    parquet_file_count = 0
    station_ids_seen = set()

    chunk_iter = pd.read_csv(
        gz_file,
        names=BY_YEAR_COLUMNS,
        header=None,
        dtype=DTYPE_MAP,
        chunksize=chunk_rows,
        low_memory=False,
    )

    for i, chunk in enumerate(chunk_iter):
        chunk = chunk.loc[chunk["element"] == "SNWD"].copy()
        if chunk.empty:
            continue

        chunk = normalize_flags(chunk)

        if DROP_MISSING:
            chunk = chunk.loc[chunk["data_value"].notna() & (chunk["data_value"] != -9999)].copy()

        if DROP_QFLAGGED:
            chunk = chunk.loc[chunk["qflag"] == ""].copy()

        if chunk.empty:
            continue

        chunk["date"] = pd.to_datetime(chunk["date_str"], format="%Y%m%d", errors="coerce")
        chunk = chunk.loc[chunk["date"].notna()].copy()

        chunk["year"] = chunk["date"].dt.year.astype("int16")
        chunk["month"] = chunk["date"].dt.month.astype("int8")
        chunk["day"] = chunk["date"].dt.day.astype("int8")
        chunk["snwd_mm"] = chunk["data_value"].astype("float32")

        keep_cols = [
            "station_id", "date", "year", "month", "day",
            "snwd_mm", "mflag", "qflag", "sflag", "obs_time"
        ]
        chunk = chunk[keep_cols]

        out_file = part_dir / f"part-{year}-{i:04d}.parquet"
        chunk.to_parquet(out_file, index=False)

        parquet_file_count += 1
        total_rows_written += len(chunk)
        station_ids_seen.update(chunk["station_id"].dropna().unique().tolist())

        if verbose:
            print(f"year={year} chunk={i:04d} rows={len(chunk):,}")

    return {
        "year": year,
        "status": "written",
        "rows_written": total_rows_written,
        "stations_written": len(station_ids_seen),
        "files": parquet_file_count,
    }

## Run the build

This may take a while for the full 1998–present run.  
The good part is that it is **restartable** by year because each year is written separately.

In [ ]:
build_log = []
for year in range(START_YEAR, END_YEAR + 1):
    result = process_one_year(year)
    build_log.append(result)
    print(result)

build_log_df = pd.DataFrame(build_log)
build_log_df

## Consolidated view of the built dataset

Pandas can read the partitioned Parquet dataset directly.

In [ ]:
snwd_all = pd.read_parquet(PARQUET_DIR)
snwd_all.head()

In [ ]:
print("Rows:", f"{len(snwd_all):,}")
print("Unique stations:", f"{snwd_all['station_id'].nunique():,}")
print("Date range:", snwd_all['date'].min(), "to", snwd_all['date'].max())

## Join station metadata

This gives you a ready-to-query station observation table with lat/lon/elevation.

In [ ]:
snwd_with_meta = snwd_all.merge(
    stations[["station_id", "lat", "lon", "elev", "state", "name", "gsn_flag", "hcn_crn_flag", "wmo_id"]],
    on="station_id",
    how="left"
)

snwd_with_meta.head()

## Build compact coverage summaries

These are very useful for deciding what subset to compare with GEOSldas later.

In [ ]:
station_coverage = (
    snwd_with_meta
    .groupby("station_id", as_index=False)
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        n_days=("date", "count"),
        max_snwd_mm=("snwd_mm", "max"),
        lat=("lat", "first"),
        lon=("lon", "first"),
        elev=("elev", "first"),
        state=("state", "first"),
        name=("name", "first"),
        gsn_flag=("gsn_flag", "first"),
        hcn_crn_flag=("hcn_crn_flag", "first"),
        wmo_id=("wmo_id", "first"),
    )
)

station_coverage["first_year"] = station_coverage["first_date"].dt.year
station_coverage["last_year"] = station_coverage["last_date"].dt.year

station_coverage = station_coverage.sort_values(["n_days", "max_snwd_mm"], ascending=[False, False])

station_coverage.head()

In [ ]:
year_summary = (
    snwd_with_meta
    .groupby("year", as_index=False)
    .agg(
        n_rows=("date", "count"),
        n_stations=("station_id", "nunique"),
        max_snwd_mm=("snwd_mm", "max"),
    )
)

year_summary

In [ ]:
coverage_file = OUTDIR / "ghcn_snwd_station_coverage.parquet"
year_summary_file = OUTDIR / "ghcn_snwd_year_summary.csv"
build_log_file = OUTDIR / "ghcn_snwd_build_log.csv"

station_coverage.to_parquet(coverage_file, index=False)
year_summary.to_csv(year_summary_file, index=False)
build_log_df.to_csv(build_log_file, index=False)

print(coverage_file)
print(year_summary_file)
print(build_log_file)

## Write a manifest

This is basic provenance so later you know exactly what was built.

In [ ]:
manifest = {
    "dataset": "NOAA/NCEI GHCN-Daily SNWD subset",
    "element": "SNWD",
    "units": "mm",
    "start_year": START_YEAR,
    "end_year": END_YEAR,
    "drop_qflagged": DROP_QFLAGGED,
    "drop_missing": DROP_MISSING,
    "source_base_url": BASE_URL,
    "build_time_utc": datetime.now(timezone.utc).isoformat(),
    "parquet_dir": str(PARQUET_DIR.resolve()),
    "files": {
        "station_metadata": str(station_meta_file.resolve()),
        "inventory": str(station_inventory_file.resolve()),
        "station_coverage": str(coverage_file.resolve()),
        "year_summary": str(year_summary_file.resolve()),
        "build_log": str(build_log_file.resolve()),
    },
}

manifest_file = OUTDIR / "ghcn_snwd_manifest.json"
with open(manifest_file, "w") as f:
    json.dump(manifest, f, indent=2)

manifest_file

## Suggested downstream use with GEOSldas

I would **not** merge this with GEOSldas in this notebook. Keep this notebook as the clean observation-build stage.

A good next-stage comparison workflow would be:

1. Read `ghcn_snwd_station_coverage.parquet`
2. Choose stations / years / seasons of interest
3. Map stations to GEOSldas grid cells in a separate notebook or module
4. Read only the needed Parquet partitions and station subsets
5. Compute station-by-station metrics there

This separation is worth it. It keeps the expensive observation ingest independent from your model-comparison logic.

## Optional refinements

You may want to add these later:

- keep a second version with **all QFLAGs retained**
- derive `snow_present = snwd_mm > 0`
- split by continent or broad region for easier operational use
- export a DuckDB database on top of the Parquet store
- create a station-to-GEOS lookup table once and reuse it